# MPP Aluminium — pathway plots

Two figures:
1. Process emissions intensity (tCO₂e/tAl, no electricity) per bucket, 2020–2050.
2. Effective power emissions intensity smelters are exposed to (MPP) vs SBTi power pathway (gCO₂/kWh).

Input: `mpp_al_intensity_pathway_1p5DS.csv` produced by notebook 03.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import PchipInterpolator

DATA_DIR = Path(".")

def smooth_line(x, y, n=300):
    xs = np.linspace(x.min(), x.max(), n)
    return xs, PchipInterpolator(x, y)(xs)

BUCKET_COLORS = {"AE": "#1f77b4", "EMDE": "#d62728", "China": "#2ca02c"}

In [ ]:
pathway = pd.read_csv(DATA_DIR / "mpp_al_intensity_pathway_1p5DS.csv")

# SBTi power pathway lives one level up in Power_Standard/. Try both cwd cases.
sbti_path = DATA_DIR / ".." / "Power_Standard" / "POWER_PATHWAY_FINAL_V2.0.csv"
if not sbti_path.exists():
    sbti_path = DATA_DIR / "Power_Standard" / "POWER_PATHWAY_FINAL_V2.0.csv"
sbti = pd.read_csv(sbti_path)

# SBTi wide → long, keep only gCO2/kWh intensity rows
year_cols = [c for c in sbti.columns if c.isdigit()]
sbti_int = sbti[sbti["Variable"].str.contains("Gross Emissions Intensity", na=False)]
sbti_long = sbti_int.melt(id_vars=["Region","Variable","Unit"], value_vars=year_cols,
                          var_name="year", value_name="gCO2_per_kWh")
sbti_long["year"] = sbti_long["year"].astype(int)
sbti_long = sbti_long[sbti_long["year"] <= 2050]
sbti_long.head()

In [ ]:
# ---- Plot 1: process intensity by bucket ----
fig, ax = plt.subplots(figsize=(9, 5.5))
for bucket, sub in pathway.groupby("bucket"):
    sub = sub.sort_values("year")
    xs, ys = smooth_line(sub["year"].values, sub["process_intensity"].values)
    ax.plot(xs, ys, color=BUCKET_COLORS[bucket], linewidth=2.2, label=bucket)
    ax.scatter(sub["year"].values[::5], sub["process_intensity"].values[::5],
               color=BUCKET_COLORS[bucket], s=18, zorder=3)

ax.set_title("MPP 1.5DS — process emissions intensity of primary aluminium\n(excluding electricity: smelter process CO₂ + PFCs + anode + refinery thermal)",
             fontsize=11)
ax.set_xlabel("Year"); ax.set_ylabel("tCO₂e / t Al")
ax.set_xlim(2020, 2050); ax.set_ylim(0, None)
ax.grid(True, alpha=0.3); ax.legend(loc="upper right", frameon=False)
plt.tight_layout(); plt.savefig(DATA_DIR / "fig_process_intensity_pathway.png", dpi=150)
plt.show()

In [ ]:
# ---- Plot 2: MPP power intensity vs SBTi power pathway ----
fig, ax = plt.subplots(figsize=(9, 5.5))

# MPP lines (solid) — effective power EF smelters actually face
for bucket, sub in pathway.groupby("bucket"):
    sub = sub.sort_values("year")
    xs, ys = smooth_line(sub["year"].values, sub["power_intensity_gCO2_per_kWh"].values)
    ax.plot(xs, ys, color=BUCKET_COLORS[bucket], linewidth=2.2, label=f"MPP Al — {bucket}")

# SBTi lines (dashed) — gross power sector intensity target
sbti_map = {"Advanced Economies": ("AE", BUCKET_COLORS["AE"]),
            "Emerging Economies": ("EMDE", BUCKET_COLORS["EMDE"])}
for region, sub in sbti_long.groupby("Region"):
    if region not in sbti_map: continue
    bucket_label, color = sbti_map[region]
    sub = sub.sort_values("year")
    xs, ys = smooth_line(sub["year"].values, sub["gCO2_per_kWh"].values)
    ax.plot(xs, ys, color=color, linewidth=2.0, linestyle="--", label=f"SBTi power — {bucket_label}")

ax.set_title("Power emissions intensity: MPP aluminium 1.5DS vs SBTi power pathway",
             fontsize=11)
ax.set_xlabel("Year"); ax.set_ylabel("gCO₂ / kWh")
ax.set_xlim(2020, 2050); ax.set_ylim(0, None)
ax.grid(True, alpha=0.3); ax.legend(loc="upper right", frameon=False, ncol=1, fontsize=9)
plt.tight_layout(); plt.savefig(DATA_DIR / "fig_power_intensity_comparison.png", dpi=150)
plt.show()